# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing all schema entities by their `@id` fields for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display basic dataset details
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into one or more record sets (tables/arrays of records). Each record set has a unique `@id` and contains fields (columns) which also have unique `@id`s.

Let's enumerate all record sets in this dataset. For each record set, we will also list their field definitions.

In [ ]:
# Helper to pretty-print Croissant schema
import json

record_sets = dataset.metadata.recordSet
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f"  Name       : {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    fields = rs.get('field', [])
    print(f"  Fields (@id):")
    for field in fields:
        print(f"    - {field.get('@id')}: {field.get('name', '')}")
    print()

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis. Each record set and field is referenced by `@id`. You may select those most relevant to the analysis.


In [ ]:
# Extract data from all record sets into DataFrames
rcset_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
dataframes = {}

for record_set_id in rcset_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id} ({len(df)} rows, {len(df.columns)} columns)")
        print(f"Columns: {df.columns.tolist()[:6]}{'...' if len(df.columns)>6 else ''}")
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")

# Choose a main record set for subsequent analysis (customize as needed)
main_record_set_id = rcset_ids[0] if rcset_ids else None
if main_record_set_id:
    print(f"\nAvailable columns in '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply initial data processing steps such as filtering, normalization, and grouping using field `@id`s.

Let's pick a numeric field (e.g., protein abundance or molecular weight) and a group field (e.g., a protein category or accession) to demonstrate typical EDA operations.


In [ ]:
# Replace these with actual field IDs from the schema above
numeric_field_id = None
group_field_id = None

# Try to auto-detect a numeric field and a suitable group field
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Heuristically choose a numeric field (e.g., containing 'MW' or 'abundance')
    for col in df.columns:
        if any(x in col.lower() for x in ['mw', 'mass', 'abundance', 'amount', 'count', 'coverage', 'percent']):
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if numeric_field_id is None:
        # Try force conversion if fields are string
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field_id = col
                    break
            except:
                continue
    # Heuristically choose a group field
    for col in df.columns:
        if any(x in col.lower() for x in ['category', 'type', 'accession', 'group', 'id']):
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
    if numeric_field_id:
        print(f"Using numeric field for EDA: {numeric_field_id}")
    else:
        print("No numeric field found for EDA.")
    if group_field_id:
        print(f"Using group field: {group_field_id}\n")
else:
    print('No main record set loaded!')

# If we found a numeric field, do basic EDA
if numeric_field_id and main_record_set_id:
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in '{main_record_set_id}' with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-9)
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
        print(f"\nTop 5 {group_field_id} group averages by '{numeric_field_id}':")
        display(grouped_df.head())
else:
    print('Unable to perform numeric analysis. Please set numeric_field_id and group_field_id to suitable column @ids as needed.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, we plot the distribution of the selected numeric field and (if applicable) the group averages.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distributions if available
if numeric_field_id and main_record_set_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Plot group averages if group_field_id found
    if group_field_id and group_field_id in df.columns and df[group_field_id].nunique() < 50:
        plt.figure(figsize=(10, 5))
        sns.barplot(y=grouped_df[group_field_id], x=grouped_df[numeric_field_id], palette="Blues_d")
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(group_field_id)
        plt.show()
else:
    print("No suitable numeric or group field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a Croissant FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles using the `mlcroissant` Python library.

Key steps included:
- Automatic inspection of available record sets and fields via their `@id`.
- Dynamic extraction of all available data tables.
- Heuristic selection and basic statistics/normalization of quantitative features.
- Grouped analysis and visualizations to uncover data patterns.

For more detailed or domain-specific analyses, refer to the dataset's Croissant schema documentation and manually specify record set and field `@id`s as needed according to your research question.
